In [7]:
# =============================================================================
# CELL 1 – IMPORTS, DARK THEME, AND HELPERS
# =============================================================================

import importlib
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown

# -------------------- Dark Theme --------------------
display(HTML("""
<style>
    body, .jp-Notebook, .jp-OutputArea-output, .jp-RenderedHTMLCommon {
        background-color: #1e1e1e !important;
        color: #d4d4d4 !important;
    }
    h2, h3, h4 {
        color: #4fc3f7 !important;
        border-bottom: 2px solid #3498db !important;
        padding-bottom: 4px;
    }
    b, strong { color: #f48fb1 !important; }
    .highlight {
        background-color: #2d2d2d !important;
        padding: 10px;
        border-left: 4px solid #3498db;
        margin: 4px 0;
        color: #d4d4d4;
    }
    code {
        background-color: #333 !important;
        color: #ffcc80 !important;
        padding: 2px 4px;
        border-radius: 4px;
    }
    .dataframe {
        background-color: #2d2d2d !important;
        color: #d4d4d4 !important;
    }
</style>
"""))

# -------------------- Verbosity Flags --------------------
SHOW_VERBOSE = True
SHOW_INFO = True
SHOW_CRITICAL = True
SHOW_DEBUG = True
# For demonstration, we keep them False to reduce output; you can set True as needed.


# -------------------- Import Extraction Module --------------------
import Extraction as Extraction6
importlib.reload(Extraction6)

# -------------------- Helper Functions --------------------
def display_title(title: str):
    display(HTML(f"<h2>{title}</h2>"))

def display_info(message: str):
    display(HTML(f"<div class='highlight'>{message}</div>"))
    

def discover_extractions(exp_root: Path) -> pd.DataFrame:
    rows = []
    for f in exp_root.rglob("extraction.json"):
        parts = f.relative_to(exp_root).parts
        try:
            models_idx = parts.index("models")
            datasets_idx = parts.index("datasets")
            model = "/".join(parts[models_idx+1:datasets_idx])
            dataset = parts[datasets_idx+1]
            rows.append({"model": model, "dataset": dataset, "path": str(f)})
        except ValueError:
            rows.append({"model": None, "dataset": None, "path": str(f)})
    return pd.DataFrame(rows)

def load_experiment_results(exp_root: Path) -> pd.DataFrame:
    """
    Load all extraction results from the experiment directory, inferring model and
    dataset names from the file path if they are missing in the JSON.
    """
    records = []
    exp_root = Path(exp_root)
    if not exp_root.exists():
        return pd.DataFrame()

    for meta_file in exp_root.rglob("extraction.json"):
        try:
            with open(meta_file, "r") as f:
                meta = json.load(f)

            # Extract model and dataset from path (most reliable)
            parts = meta_file.relative_to(exp_root).parts
            try:
                models_idx = parts.index("models")
                datasets_idx = parts.index("datasets")
                model = "/".join(parts[models_idx+1:datasets_idx])
                dataset = parts[datasets_idx+1]
            except ValueError:
                # Fallback to metadata
                model = meta.get("model", {}).get("name")
                dataset = meta.get("dataset", {}).get("name")

            record = {
                "experiment_id": meta.get("experiment_id"),
                "model": model,
                "dataset": dataset,
                "status": meta.get("status"),
                "completed_samples": meta.get("performance", {}).get("completed_samples"),
                "total_samples": meta.get("dataset", {}).get("samples"),
                "batch_size": meta.get("extraction", {}).get("batch_size"),
                "pooling": meta.get("extraction", {}).get("pooling"),
                "max_length": meta.get("extraction", {}).get("max_length"),
                "samples_per_second": meta.get("performance", {}).get("samples_per_second"),
                "tokens_per_second": meta.get("performance", {}).get("tokens_per_second"),
                "elapsed_seconds": meta.get("performance", {}).get("elapsed_seconds"),
                "error": None,
                "text_column": meta.get("dataset", {}).get("text_column"),
                "label_column": meta.get("dataset", {}).get("labels", {}).get("label_column"),
            }
            records.append(record)
        except Exception:
            continue

    return pd.DataFrame(records)

print("Environment ready. Dark theme applied.")

Environment ready. Dark theme applied.


In [8]:
# =============================================================================
# CELL 2 — CANONICAL DATASET DISCOVERY (drop-in, no registry module)
# =============================================================================

from pathlib import Path
import importlib
import Extraction as EX
importlib.reload(EX)

display_title("Processed dataset discovery")

DATASETS, CSV_HASHES = EX.discover_processed_datasets(
    datasets_root=EX.PROCESSED_DATASETS_ROOT,
    show_info=True,
)

display_info(
    f"<b>{len(DATASETS)}</b> processed datasets discovered under "
    f"<code>{EX.PROCESSED_DATASETS_ROOT}</code>"
)

# Visual summary
summary = pd.DataFrame([
    {
        "name": name,
        "rows": len(df),
        "columns": ", ".join(df.columns),
        "csv_sha256": CSV_HASHES[name][:16] + "…",
        "sample_label": str(df["label"].iloc[0]),
    }
    for name, df in DATASETS.items()
])
display(summary)

# Contract checks
for name, df in DATASETS.items():
    assert list(df.columns) == list(EX.PROCESSED_COLUMNS), f"{name}: bad columns"
    assert df.index.is_unique and df.index[0] == 0 and df.index[-1] == len(df) - 1, \
        f"{name}: index must be a clean RangeIndex so row i ↔ hidden_states[i]"
    assert df["label"].map(lambda x: isinstance(x, list) and len(x) > 0).all(), \
        f"{name}: all labels must be non-empty Python lists"

display_info("✅ All datasets satisfy the extraction contract.")

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# CELL 3 — RUN MODEL MATRIX ON CLEAN DATA
# =============================================================================

# The pipeline writes to a single fixed root:
#     /Volumes/Amirali/Probing-Emotions/
# `base_output` and `auto_batch_size` no longer exist as arguments.
# Ordering is controlled by /Volumes/Amirali/Probing-Emotions/extraction_order.json.

NEW_EXPERIMENT_ID = "master_v1"

results = EX.run_model_matrix(
    datasets=DATASETS,
    dataset_csv_hashes=CSV_HASHES,
    groups=None,
    experiment_id=NEW_EXPERIMENT_ID,
    pooling="mean",
    max_length=512,
    use_half_precision=True,
    flush_every_batches=8,
    continue_on_model_error=True,
    show_verbose=True,
    show_info=True,
    show_critical=True,
    show_debug=True,
)

print(f"Result records : {len(results)}")
print(f"Output root    : {EX.EXTERNAL_ROOT}")
print(f"HF cache       : {EX.HF_HUB_CACHE}")


╔══════════════════════════════════════════════════════════════════════════════════════╗
║ MODEL MATRIX                                                                         ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
  Requested experiment : master_v1
  Transformers         : 4.57.6
  PyTorch              : 2.2.2
  Models               : 25
  Datasets             : 6
  Output root          : /Volumes/Amirali/Probing-Emotions
01. [01_encoders] BERT         0.11B  google-bert/bert-base-uncased
02. [01_encoders] DistilBERT   0.066B  distilbert/distilbert-base-uncased
03. [01_encoders] RoBERTa      0.125B  FacebookAI/roberta-base
04. [01_encoders] ELECTRA      0.014B  google/electra-small-discriminator
05. [01_encoders] DeBERTa      0.14B  microsoft/deberta-v3-small
06. [02_early_decoders] GPT          0.124B  gpt2
07. [02_early_decoders] GPT-Neo      0.125B  EleutherAI/gpt-neo-125m
08. [02_early_decoders] OPT          0.125B  facebook/opt

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
MODEL PREPARATION FAILURE — attempt 1/4
  Model                : distilbert/distilbert-base-uncased
  Revision             : 12040accade4e8a0f71eabdb258fecc2e7e948be
  Error type           : LocalEntryNotFoundError
  Transient            : True
  Error                : An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. Please check your connection and try again or make sure your Internet connection is on.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
  Traceback:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/huggingface_hub/file_download.py", line 1572, in _get_metadata_or_catch_error
    raise FileMetadataError(
huggingface_hub.errors.FileMetadataError: Distant resource does not seem to be on huggingface.co. It

KeyboardInterrupt: 

### How to load the missing QWENs ?  

In [ ]:
# =============================================================================
# CELL 4 — SUMMARY FROM RUNS/CLEAN_V1
# =============================================================================

RUN_ROOT = EXTERNAL_ROOT / "runs" / NEW_EXPERIMENT_ID
df_results = load_experiment_results(RUN_ROOT)

if not df_results.empty:
    display_title("Extraction progress")
    df_show = df_results.copy()
    df_show["model"] = df_show["model"].str.replace("__", "/", regex=False)
    display(df_show[["model", "dataset", "status", "completed_samples", "total_samples"]])
    display_info(f"Pairs discovered: <b>{len(df_results)}</b>")
else:
    display_info("No extraction metadata yet — run CELL 3.")

In [ ]:
from sys import audit


df_audit = pd.DataFrame(audit)
if not df_audit.empty:
    # Primary contract: extraction's n_samples must equal the processed CSV's
    # row count. Otherwise the memmap is misaligned with the source data.
    def _csv_len(dataset_name):
        return len(DATASETS.get(dataset_name, []))
    df_audit["csv_rows"] = df_audit["dataset_name"].map(_csv_len)
    df_audit["csv_matches"] = df_audit["n_samples"] == df_audit["csv_rows"]
    df_audit["row_count_ok"] = df_audit["n_samples"] == df_audit["completed_count"]

    display(df_audit[[
        "model_name", "dataset_name", "n_samples", "csv_rows", "completed_count",
        "csv_matches", "row_count_ok",
        "status", "checksum_match", "sample_ids_match",
    ]])

In [ ]:
# =============================================================================
# CELL 5 – VISUALISATIONS & ANALYSIS 
# =============================================================================

if not df_results.empty:
    # Prepare data for plotting
    df = df_results.copy()
    df['completion_pct'] = df['completed_samples'] / df['total_samples'] * 100
    df['status_clean'] = df['status'].replace({'complete': 'Complete', 'partial': 'Partial', 'failed': 'Failed', 'already_complete': 'Already Complete'})

    sns.set_style("darkgrid")
    plt.rcParams.update({
        'figure.facecolor': '#1e1e1e',
        'axes.facecolor': '#2d2d2d',
        'axes.edgecolor': '#d4d4d4',
        'axes.labelcolor': '#d4d4d4',
        'text.color': '#d4d4d4',
        'xtick.color': '#d4d4d4',
        'ytick.color': '#d4d4d4',
        'grid.color': '#444444',
        'legend.facecolor': '#2d2d2d',
        'legend.edgecolor': '#d4d4d4',
    })

    # ---- 1. Completion status per model/dataset ----
    fig, ax = plt.subplots(figsize=(12, 8))
    pivot = df.pivot_table(index='model', columns='dataset', values='completion_pct', aggfunc='max')
    sns.heatmap(pivot, annot=True, fmt=".0f", cmap="viridis", cbar_kws={'label': 'Completion %'}, ax=ax)
    ax.set_title('Completion Percentage by Model and Dataset', color='#4fc3f7')
    plt.tight_layout()
    plt.show()

    # ---- 2. Throughput (samples/sec) by model ----
    df_complete = df[df['samples_per_second'].notna()]
    if not df_complete.empty:
        fig, ax = plt.subplots(figsize=(14, 6))
        sns.barplot(data=df_complete, x='model', y='samples_per_second', hue='dataset', palette='coolwarm', ax=ax)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.set_title('Extraction Throughput (samples/sec)', color='#4fc3f7')
        plt.tight_layout()
        plt.show()

    # ---- 3. Total time per model ----
    df_time = df.groupby('model')['elapsed_seconds'].sum().reset_index().sort_values('elapsed_seconds', ascending=False)
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(data=df_time, x='elapsed_seconds', y='model', palette='magma', ax=ax)
    ax.set_xlabel('Total Elapsed Time (seconds)')
    ax.set_title('Cumulative Extraction Time per Model', color='#4fc3f7')
    plt.tight_layout()
    plt.show()

    # ---- 4. Label coverage for completed datasets ----
    # We'll just display label column info
    display_title("Label Columns Used")
    display(df[['model', 'dataset', 'label_column']].drop_duplicates())
else:
    display_info("No data to visualise. Please run extraction first.")

In [ ]:
# =============================================================================
# CELL 6 – FINAL SUMMARY & EXPORT
# =============================================================================

if not df_results.empty:
    # Save consolidated CSV
    output_csv = exp_root / "extraction_summary.csv"
    df_results.to_csv(output_csv, index=False)
    display_title("Final Report")
    display(df_results)
    display_info(f"Report exported to <code>{output_csv}</code>")
else:
    display_info("Nothing to export yet.")

In [ ]:
discover_extractions(exp_root)

In [ ]:
# Check which models have at least one dataset complete
model_status = df_results.groupby("model")["dataset"].nunique()
print(model_status)